# Evolve Guidance Evaluation

This notebook follows the DAS-style evaluation workflow:

- load generated images from one or more experiment folders
- rescore them with reward models for cross-reward generalization
- compute LPIPS-based sample diversity
- compare methods with tables, plots, and quick image grids

It is intended for outputs such as `logs/.../eval_vis/*.png` produced by `DAS.py` or other batch runs.

In [ ]:
from __future__ import annotations

import itertools
import math
import random
import sys
from collections import defaultdict
from pathlib import Path

import lpips
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from IPython.display import display

try:
    import pandas as pd
except ImportError:
    pd = None


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'setup.py').exists() and (candidate / 'seg').exists():
            return candidate
    raise RuntimeError('Could not locate project root from notebook location.')


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'DEVICE = {DEVICE}')
print(f'DTYPE = {DTYPE}')

## Configure Experiments

Point each entry to a folder containing generated images. For DAS-style runs, the usual target is an `eval_vis` directory.

If your filenames follow the DAS pattern `000_prompt text | reward: ...png`, the notebook will automatically recover prompts for text-conditioned rescoring.

In [ ]:
EXPERIMENTS = {
    'sd_test_1': PROJECT_ROOT / 'logs' / 'sd' / 'test_1',
    # 'baseline': PROJECT_ROOT / 'logs' / 'DiffusionSampleBaseline' / 'pick' / 'baseline_run' / 'eval_vis',
    # 'stein': PROJECT_ROOT / 'logs' / 'DiffusionSampleBaseline' / 'pick' / 'stein_run' / 'eval_vis',
}

IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.webp', '.bmp'}
SAMPLE_IMAGE_PREFIXES = ('sample_',)
SCORERS_TO_RUN = ['pick', 'clip', 'image_reward', 'aesthetic']
BATCH_SIZE = 4
EVAL_IMAGE_SIZE = (512, 512)
LPIPS_NET = 'alex'
LPIPS_IMAGE_SIZE = (256, 256)
LPIPS_MAX_PAIRS = 256
GRID_IMAGES_PER_METHOD = 4
RANDOM_SEED = 42

assert EXPERIMENTS, 'Fill in EXPERIMENTS before running the evaluation cells.'
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

In [ ]:
from seg.scorers.ImageReward_scorer import ImageRewardScorer
from seg.scorers.PickScore_scorer import PickScoreScorer
from seg.scorers.clip_scorer import CLIPScorer
from seg.scorers.aesthetic_scorer import AestheticScorer


def parse_prompt_from_filename(path: Path) -> str | None:
    stem = path.stem
    if ' | reward:' in stem and '_' in stem:
        prefix = stem.split(' | reward:', 1)[0]
        if '_' in prefix:
            return prefix.split('_', 1)[1]

    for parent in path.parents:
        if parent.name.startswith('run_'):
            parts = parent.name.split('_', 2)
            if len(parts) == 3:
                return parts[2].replace('-', ' ')
    return None


def is_candidate_image(path: Path) -> bool:
    if path.suffix.lower() not in IMAGE_EXTS:
        return False
    if path.parent.name == 'eval_vis':
        return True
    return path.name.startswith(SAMPLE_IMAGE_PREFIXES)


def collect_records(experiments: dict[str, Path]) -> list[dict]:
    records = []
    for method, folder in experiments.items():
        folder = Path(folder)
        if not folder.exists():
            raise FileNotFoundError(f'Experiment folder not found: {folder}')
        image_paths = sorted(p for p in folder.rglob('*') if is_candidate_image(p))
        if not image_paths:
            raise FileNotFoundError(f'No images found under: {folder}')
        for image_path in image_paths:
            records.append({
                'method': method,
                'image_path': image_path,
                'prompt': parse_prompt_from_filename(image_path),
            })
    return records


def load_image_tensor(image_paths: list[Path]) -> torch.Tensor:
    images = []
    for image_path in image_paths:
        with Image.open(image_path) as image:
            image = image.convert('RGB').resize(EVAL_IMAGE_SIZE, Image.BICUBIC)
            array = np.asarray(image, dtype=np.float32) / 255.0
        tensor = torch.from_numpy(array).permute(2, 0, 1)
        images.append(tensor)
    return torch.stack(images, dim=0)


def batched(items: list, batch_size: int):
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]


def build_scorers(names: list[str], device: str, dtype: torch.dtype) -> dict[str, object]:
    scorers = {}
    for name in names:
        if name == 'pick':
            scorers[name] = PickScoreScorer(dtype=dtype, device=device)
        elif name == 'clip':
            scorers[name] = CLIPScorer(dtype=dtype, device=device)
        elif name == 'image_reward':
            scorers[name] = ImageRewardScorer(dtype=dtype, device=device)
        elif name == 'aesthetic':
            scorers[name] = AestheticScorer(dtype=dtype, device=device)
        else:
            raise ValueError(f'Unsupported scorer: {name}')
    return scorers


def score_batch(scorer_name: str, scorer: object, images: torch.Tensor, prompts: list[str | None]) -> torch.Tensor:
    images = images.to(device=DEVICE)
    if scorer_name == 'aesthetic':
        return scorer(images).detach().float().cpu()
    if any(prompt is None for prompt in prompts):
        raise ValueError(
            f'Scorer {scorer_name} requires prompts, but some filenames did not contain DAS-style prompts.'
        )
    return scorer(images, prompts).detach().float().cpu()


def summarize_rows(rows: list[dict], sort_by: str | None = None):
    if sort_by is not None:
        rows = sorted(rows, key=lambda row: row[sort_by], reverse=True)
    if pd is not None:
        display(pd.DataFrame(rows))
    else:
        for row in rows:
            print(row)

## Load Records

In [ ]:
records = collect_records(EXPERIMENTS)
records_by_method = defaultdict(list)
for record in records:
    records_by_method[record['method']].append(record)

dataset_rows = []
for method, method_records in records_by_method.items():
    prompt_count = sum(record['prompt'] is not None for record in method_records)
    dataset_rows.append({
        'method': method,
        'num_images': len(method_records),
        'num_prompt_parsed': prompt_count,
        'folder': str(EXPERIMENTS[method]),
    })

summarize_rows(dataset_rows)
print(f'Total images loaded: {len(records)}')

## Cross-Reward Generalization

This rescoring step mirrors the DAS evaluation idea: a method optimized for one reward should ideally still score well under other reward models.

In [ ]:
scorers = build_scorers(SCORERS_TO_RUN, device=DEVICE, dtype=DTYPE)

for scorer_name, scorer in scorers.items():
    for batch in batched(records, BATCH_SIZE):
        image_paths = [record['image_path'] for record in batch]
        prompts = [record['prompt'] for record in batch]
        images = load_image_tensor(image_paths)
        scores = score_batch(scorer_name, scorer, images, prompts)
        for record, score in zip(batch, scores.tolist()):
            record[f'score_{scorer_name}'] = float(score)

summary_rows = []
for method, method_records in records_by_method.items():
    row = {'method': method, 'num_images': len(method_records)}
    for scorer_name in SCORERS_TO_RUN:
        values = [record[f'score_{scorer_name}'] for record in method_records]
        row[f'{scorer_name}_mean'] = float(np.mean(values))
        row[f'{scorer_name}_std'] = float(np.std(values))
    summary_rows.append(row)

summarize_rows(summary_rows)

In [ ]:
fig, axes = plt.subplots(1, len(SCORERS_TO_RUN), figsize=(5 * len(SCORERS_TO_RUN), 4), squeeze=False)
axes = axes[0]
methods = list(records_by_method.keys())

for ax, scorer_name in zip(axes, SCORERS_TO_RUN):
    means = [np.mean([record[f'score_{scorer_name}'] for record in records_by_method[method]]) for method in methods]
    stds = [np.std([record[f'score_{scorer_name}'] for record in records_by_method[method]]) for method in methods]
    ax.bar(methods, means, yerr=stds, capsize=4)
    ax.set_title(f'{scorer_name} score')
    ax.tick_params(axis='x', rotation=30)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Sample Diversity With LPIPS

This computes average pairwise LPIPS inside each method. If a method has many images, the notebook samples up to `LPIPS_MAX_PAIRS` pairs for speed.

In [ ]:
lpips_model = lpips.LPIPS(net=LPIPS_NET).to(DEVICE)
lpips_model.eval()


def load_lpips_image(path: Path) -> torch.Tensor:
    with Image.open(path) as image:
        image = image.convert('RGB').resize(LPIPS_IMAGE_SIZE, Image.BICUBIC)
        array = np.asarray(image, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(array).permute(2, 0, 1)
    return (tensor * 2.0 - 1.0).unsqueeze(0)


def pairwise_lpips(paths: list[Path], max_pairs: int) -> float:
    if len(paths) < 2:
        return float('nan')

    all_pairs = list(itertools.combinations(paths, 2))
    if len(all_pairs) > max_pairs:
        all_pairs = random.sample(all_pairs, max_pairs)

    values = []
    with torch.no_grad():
        for left_path, right_path in all_pairs:
            left = load_lpips_image(left_path).to(DEVICE)
            right = load_lpips_image(right_path).to(DEVICE)
            value = lpips_model(left, right).item()
            values.append(value)
    return float(np.mean(values))


diversity_rows = []
for method, method_records in records_by_method.items():
    diversity_rows.append({
        'method': method,
        'num_images': len(method_records),
        'lpips_diversity': pairwise_lpips([record['image_path'] for record in method_records], LPIPS_MAX_PAIRS),
    })

summarize_rows(diversity_rows, sort_by='lpips_diversity')

In [ ]:
methods = [row['method'] for row in diversity_rows]
values = [row['lpips_diversity'] for row in diversity_rows]

plt.figure(figsize=(6, 4))
plt.bar(methods, values)
plt.title('LPIPS Diversity')
plt.grid(alpha=0.3)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Quick Image Grid

In [ ]:
num_methods = len(records_by_method)
fig, axes = plt.subplots(num_methods, GRID_IMAGES_PER_METHOD, figsize=(4 * GRID_IMAGES_PER_METHOD, 4 * num_methods))

if num_methods == 1:
    axes = np.expand_dims(axes, axis=0)

for row_idx, (method, method_records) in enumerate(records_by_method.items()):
    chosen = method_records[:GRID_IMAGES_PER_METHOD]
    for col_idx in range(GRID_IMAGES_PER_METHOD):
        ax = axes[row_idx, col_idx]
        ax.axis('off')
        if col_idx >= len(chosen):
            continue
        record = chosen[col_idx]
        with Image.open(record['image_path']) as image:
            ax.imshow(image.convert('RGB'))
        title = method
        if record['prompt'] is not None:
            title += '\n' + record['prompt'][:60]
        ax.set_title(title, fontsize=10)

plt.tight_layout()
plt.show()